<a href="https://colab.research.google.com/github/HariniRJ/ai-lab/blob/main/Unification_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re

class Term:
    """Represents a term in First-Order Logic (Constant, Variable, or Function/Predicate)."""
    def __init__(self, name, args=None):
        self.name = name
        self.args = args if args is not None else []

    @property
    def is_variable(self):
        return len(self.args) == 0 and self.name in ['x', 'y', 'z', 'X', 'Y', 'Z', 'w', 'W']

    @property
    def is_constant(self):
        return len(self.args) == 0 and not self.is_variable

    def occurs_in(self, other, subst):
        """Occurs-check to prevent infinite recursion loop."""
        if self == other:
            return True
        if other.is_variable:
            if other.name in subst:
                return self.occurs_in(subst[other.name], subst)
            return False
        return any(self.occurs_in(arg, subst) for arg in other.args)

    def apply_subst(self, subst):
        """Applies a substitution dictionary to the term recursively."""
        if self.is_variable:
            if self.name in subst:
                return subst[self.name].apply_subst(subst)
            return self
        if self.is_constant:
            return self
        return Term(self.name, [arg.apply_subst(subst) for arg in self.args])

    def __eq__(self, other):
        if not isinstance(other, Term):
            return False
        if self.name != other.name or len(self.args) != len(other.args):
            return False
        return all(a == b for a, b in zip(self.args, other.args))

    def __str__(self):
        if not self.args:
            return self.name
        return f"{self.name}({', '.join(str(arg) for arg in self.args)})"

    def __repr__(self):
        return self.__str__()


def trace_unify(x, y):
    """Wrapper function to initialize variables and run step-by-step tracking."""
    if x.name != y.name or len(x.args) != len(y.args):
        print("Result: ❌ FAILURE (Predicate symbols or argument lengths mismatch)")
        return

    # Track structural states
    subst = {}
    step = 0

    print(f"\nS{step} => {{ {x} ; {y} }}")

    for i, (arg_x, arg_y) in enumerate(zip(x.args, y.args)):
        # Apply whatever substitution we have found up to this point
        curr_x = arg_x.apply_subst(subst)
        curr_y = arg_y.apply_subst(subst)

        if curr_x == curr_y:
            continue

        # Determine variable assignment direction
        v, t = None, None
        if curr_x.is_variable:
            v, t = curr_x, curr_y
        elif curr_y.is_variable:
            v, t = curr_y, curr_x

        if v is not None:
            # Run Occurs Check
            if v.occurs_in(t, subst):
                print(f"SUBST θ = Failed Occurs Check ({v} in {t})")
                print("Result: ❌ FAILURE")
                return

            # Record substitution step
            subst[v.name] = t
            step += 1
            print(f"SUBST θ = {{{v.name}/{t}}}")

            # Apply newest substitutions to display current step states
            print(f"S{step} => {{ {x.apply_subst(subst)} ; {y.apply_subst(subst)} }}")
        else:
            # Complex sub-term structural mismatch (e.g. f(...) vs g(...))
            # Try to recursively resolve inner functions
            inner_subst = unify(curr_x, curr_y, subst)
            if inner_subst == "FAILURE":
                print(f"SUBST θ = Failed to match sub-structures {curr_x} and {curr_y}")
                print("Result: ❌ FAILURE")
                return
            else:
                for k, v in inner_subst.items():
                    if k not in subst:
                        subst[k] = v
                        step += 1
                        print(f"SUBST θ = {{{k}/{v}}}")
                        print(f"S{step} => {{ {x.apply_subst(subst)} ; {y.apply_subst(subst)} }}")

    print("\nResult: 🎉 Unified Successfully.")
    formatted_subst = {k: str(v) for k, v in subst.items()}
    print(f"Final MGU (θ): {formatted_subst}")


def unify(x, y, subst):
    """Fallback recursive routine for nested structures."""
    if x == y: return subst
    if x.is_variable:
        if x.name in subst: return unify(subst[x.name], y, subst)
        if x.occurs_in(y, subst): return "FAILURE"
        subst[x.name] = y
        return subst
    if y.is_variable:
        if y.name in subst: return unify(x, subst[y.name], subst)
        if y.occurs_in(x, subst): return "FAILURE"
        subst[y.name] = x
        return subst
    if x.is_constant or y.is_constant: return "FAILURE"
    if x.name != y.name or len(x.args) != len(y.args): return "FAILURE"

    for ax, ay in zip(x.args, y.args):
        subst = unify(ax.apply_subst(subst), ay.apply_subst(subst), subst)
        if subst == "FAILURE": return "FAILURE"
    return subst


def parse_term(s):
    """Safely breaks down FOL strings into structural object classes."""
    s = s.replace(" ", "")
    match = re.match(r"^([A-Za-z0-9_]+)(?:\((.*)\))?$", s)
    if not match: raise ValueError(f"Syntax error on parsing token: {s}")
    name, args_str = match.groups()
    if not args_str: return Term(name)

    args = []
    bracket_count = 0
    current_arg = []
    for char in args_str:
        if char == ',' and bracket_count == 0:
            args.append(parse_term("".join(current_arg)))
            current_arg = []
        else:
            if char == '(': bracket_count += 1
            if char == ')': bracket_count -= 1
            current_arg.append(char)
    if current_arg:
        args.append(parse_term("".join(current_arg)))
    return Term(name, args)


# =====================================================================
# INTERACTIVE TERMINAL LOOP
# =====================================================================
if __name__ == "__main__":
    print("=== FIRST ORDER LOGIC UNIFICATION TERMINAL (STEP BY STEP) ===")
    print("Example layout format match: P(x, y) or p(b, x, f(g(z)))")
    print("Type 'exit' to quit.\n")

    while True:
        try:
            expr1_str = input("Enter Expression 1: ").strip()
            if expr1_str.lower() in ['exit', 'quit']: break
            if not expr1_str: continue

            expr2_str = input("Enter Expression 2: ").strip()
            if expr2_str.lower() in ['exit', 'quit']: break
            if not expr2_str: continue

            t1 = parse_term(expr1_str)
            t2 = parse_term(expr2_str)

            print("\n" + "="*50)
            trace_unify(t1, t2)
            print("="*50 + "\n")

        except Exception as e:
            print(f"\n⚠️ Format Error: Verify string tokens and parentheses. ({e})\n")


=== FIRST ORDER LOGIC UNIFICATION TERMINAL (STEP BY STEP) ===
Example layout format match: P(x, y) or p(b, x, f(g(z)))
Type 'exit' to quit.

Enter Expression 1: p(b,x,f(g(z)))
Enter Expression 2: p(z,f(y),f(y))


S0 => { p(b, x, f(g(z))) ; p(z, f(y), f(y)) }
SUBST θ = {z/b}
S1 => { p(b, x, f(g(b))) ; p(b, f(y), f(y)) }
SUBST θ = {x/f(y)}
S2 => { p(b, f(y), f(g(b))) ; p(b, f(y), f(y)) }

Result: 🎉 Unified Successfully.
Final MGU (θ): {'z': 'b', 'x': 'f(y)', 'y': 'g(b)'}

